## CSV → Embeddings → Azure AI Search

### Install dependencies

In [3]:
pip install openai azure-search-documents pandas python-dotenv

Note: you may need to restart the kernel to use updated packages.


### Imports


In [4]:
import pandas as pd
import openai
from openai import AzureOpenAI

### Load Environemnt Variables


In [20]:
from dotenv import load_dotenv
import os

load_dotenv()

OPENAI_KEY = os.getenv("AZURE_OPENAI_API_KEY")
OPENAI_ENDPOINT = os.getenv("AZURE_OPENAI_ENDPOINT")

SEARCH_ENDPOINT = os.getenv("AZURE_SEARCH_ENDPOINT")
SEARCH_KEY = os.getenv("AZURE_SEARCH_KEY")

### Load CSV

In [6]:
df = pd.read_csv("gold_layer_chunks.csv")
df.head()

,id,content,type,length
0,129,"Customer 129.0 has placed 11.0 total orders, w...",customer,180
1,713,"Customer 713.0 has placed 15.0 total orders, w...",customer,175
2,728,"Customer 728.0 has placed 18.0 total orders, w...",customer,181
3,1272,"Customer 1272.0 has placed 19.0 total orders, ...",customer,182
4,1379,"Customer 1379.0 has placed 10.0 total orders, ...",customer,176


### Configure Azure OpenAI

In [7]:
from openai import AzureOpenAI


In [8]:
client = AzureOpenAI(
    api_version="2024-02-01",
    azure_endpoint=OPENAI_ENDPOINT,
    api_key=OPENAI_KEY
)

In [9]:
print(OPENAI_ENDPOINT)

https://textembedding-3-small.openai.azure.com/


### Embedding function

In [10]:
def get_embedding(text):
    response = client.embeddings.create(
        model="text-embedding-3-small",   # deployment name
        input=text
    )
    return response.data[0].embedding

In [11]:
emb = get_embedding("test")
print(len(emb))

1536


### Generate embeddings

In [12]:
embeddings = []

for i, text in enumerate(df["content"]):
    emb = get_embedding(text)
    embeddings.append(emb)
    
    if i % 100 == 0:
        print(f"Processed {i} rows")

df["embedding"] = embeddings

Processed 0 rows
Processed 100 rows
Processed 200 rows
Processed 300 rows
Processed 400 rows
Processed 500 rows
Processed 600 rows
Processed 700 rows
Processed 800 rows
Processed 900 rows
Processed 1000 rows
Processed 1100 rows
Processed 1200 rows
Processed 1300 rows
Processed 1400 rows
Processed 1500 rows
Processed 1600 rows
Processed 1700 rows
Processed 1800 rows
Processed 1900 rows
Processed 2000 rows
Processed 2100 rows
Processed 2200 rows
Processed 2300 rows
Processed 2400 rows
Processed 2500 rows
Processed 2600 rows
Processed 2700 rows
Processed 2800 rows
Processed 2900 rows
Processed 3000 rows
Processed 3100 rows
Processed 3200 rows
Processed 3300 rows
Processed 3400 rows


### Validate embedding shape

In [15]:
len(df["embedding"][0])

1536

This cell defines a **vector-enabled search index** that allows:

- storing embeddings
- performing semantic search
- powering retrieval in a RAG pipeline

In [35]:
import requests
import json

endpoint = SEARCH_ENDPOINT
index_name = "instacart-index"
api_key = SEARCH_KEY

url = f"{endpoint}/indexes/{index_name}?api-version=2024-05-01-preview"

headers = {
    "Content-Type": "application/json",
    "api-key": api_key
}

index_schema = {
    "name": index_name,
    "fields": [
        {"name": "id", "type": "Edm.String", "key": True},
        {"name": "content", "type": "Edm.String", "searchable": True},
        {"name": "type", "type": "Edm.String", "filterable": True},

        # ✅ VECTOR FIELD
        {
            "name": "embedding",
            "type": "Collection(Edm.Single)",
            "searchable": True,
            "dimensions": 1536,
            "vectorSearchProfile": "vector-profile"
        }
    ],

    "vectorSearch": {
        "algorithms": [
            {
                "name": "hnsw-config",
                "kind": "hnsw"
            }
        ],
        "profiles": [
            {
                "name": "vector-profile",
                "algorithm": "hnsw-config"
            }
        ]
    }
}

In [36]:
response = requests.put(url, headers=headers, data=json.dumps(index_schema))

print(response.status_code)
print(response.text)

204



### Index creation in Azure AI Search

In [37]:
# Search client

from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name="instacart-index",
    credential=AzureKeyCredential(SEARCH_KEY)
)

# Prepare documents

docs = []

for _, row in df.iterrows():
    docs.append({
        "id": str(row["id"]),
        "content": row["content"],
        "type": row["type"],
        "embedding": row["embedding"]
    })

# upload in batches

batch_size = 100

for i in range(0, len(docs), batch_size):
    batch = docs[i:i+batch_size]
    search_client.upload_documents(batch)
    print(f"Uploaded {i} → {i+batch_size}")

Uploaded 0 → 100
Uploaded 100 → 200
Uploaded 200 → 300
Uploaded 300 → 400
Uploaded 400 → 500
Uploaded 500 → 600
Uploaded 600 → 700
Uploaded 700 → 800
Uploaded 800 → 900
Uploaded 900 → 1000
Uploaded 1000 → 1100
Uploaded 1100 → 1200
Uploaded 1200 → 1300
Uploaded 1300 → 1400
Uploaded 1400 → 1500
Uploaded 1500 → 1600
Uploaded 1600 → 1700
Uploaded 1700 → 1800
Uploaded 1800 → 1900
Uploaded 1900 → 2000
Uploaded 2000 → 2100
Uploaded 2100 → 2200
Uploaded 2200 → 2300
Uploaded 2300 → 2400
Uploaded 2400 → 2500
Uploaded 2500 → 2600
Uploaded 2600 → 2700
Uploaded 2700 → 2800
Uploaded 2800 → 2900
Uploaded 2900 → 3000
Uploaded 3000 → 3100
Uploaded 3100 → 3200
Uploaded 3200 → 3300
Uploaded 3300 → 3400
Uploaded 3400 → 3500
Uploaded 3500 → 3600
Uploaded 3600 → 3700
Uploaded 3700 → 3800
Uploaded 3800 → 3900
Uploaded 3900 → 4000
Uploaded 4000 → 4100
Uploaded 4100 → 4200


## Perform Vector Similarity Search (Retrieval Step)

This cell executes the **retrieval stage** of the RAG pipeline using Azure AI Search.


In [38]:
query = "What defines a loyal customer?"
query_embedding = get_embedding(query)

results = search_client.search(
    search_text="",
    vector_queries=[
        {
            "kind": "vector",
            "vector": query_embedding,
            "k": 3,
            "fields": "embedding"
        }
    ]
)

for r in results:
    print("\n--- Result ---")
    print(r["content"])


--- Result ---
Customer 58937.0 has placed 31.0 total orders, with an average basket size of 5.48 items and a reorder ratio of 0.77. This customer is a high-frequency, highly loyal shopper.

--- Result ---
Customer 101735.0 has placed 51.0 total orders, with an average basket size of 11.29 items and a reorder ratio of 0.86. This customer is a high-frequency, highly loyal shopper.

--- Result ---
Customer 49420.0 has placed 99.0 total orders, with an average basket size of 11.39 items and a reorder ratio of 0.71. This customer is a high-frequency, highly loyal shopper.


In [39]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=OPENAI_KEY,
    api_version = "2025-03-01-preview",
    azure_endpoint=OPENAI_ENDPOINT 
)


## Generate Data-Driven Answer Using GPT-5 (Generation Step)

This cell performs the **generation stage** of the RAG pipeline using a GPT-5 model.

In [40]:
context = "\n".join([r["content"] for r in results])

response = client.responses.create(
    model="gpt-5.4-mini",
    input=f"""
You are analyzing customer behavior using a dataset.

Use the context to infer patterns and derive a definition.

Do NOT give a generic definition.
Instead, base your answer on the numerical patterns in the data.

Context:
{context}

Question:
What defines a loyal customer?

Answer:
Identify patterns in the data (e.g. reorder ratio, total orders)
and derive a data-driven definition.
"""
)

print(response.output[0].content[0].text)

A **loyal customer** here is best defined as someone who shows **repeat purchasing behavior with a high share of reordered items**.

Data-driven pattern to use:
- **Total orders:** loyal customers tend to have **multiple orders**, not just a one-time purchase.
- **Reorder ratio / reordered items:** they usually have a **high proportion of products marked as reordered**.
- Often, the strongest signal is a customer with **consistent repeat orders across baskets**, rather than a single large order.

So, in this dataset, a loyal customer is one who:
1. Has **more than one order**,
2. Shows a **high reorder rate**,
3. Regularly buys items they have purchased before.

In short: **loyal customers are repeated buyers whose baskets are dominated by reorders, indicating ongoing preference rather than occasional shopping.**
